# GenCast Mini — run on a Colab GPU kernel (via VS Code)

This notebook runs the same GenCast Mini pipeline already verified on the GPU
server, but against a Colab-hosted GPU kernel connected through VS Code's
Colab extension. It clones the repo fresh onto the Colab VM's own filesystem
(separate from this file's location on your laptop), downloads the public
GenCast Mini data, runs `gencast_mini_run.py` with the same GPU attention fix
already used before, and leaves `predictions.nc` plus the sanity/error-map/
RMSE diagnostics in `gencast_output/` on the Colab VM.

**Before running:** in VS Code, select the kernel for this notebook and pick
your connected Colab GPU runtime (not a local Python kernel). Then run the
cells below top to bottom, checking each one finishes before starting the
next -- some early steps (package installs, data download) take a few
minutes.


## 1. Check the GPU

In [1]:
!nvidia-smi


Mon Oct  5 09:56:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repo onto the Colab VM

The Colab kernel runs on its own remote machine with its own filesystem -- it does not see files on your laptop. This clones a fresh copy there.

In [3]:
REPO_DIR = "/content/weathernext"

import os
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/swatibhalaik/weathernext.git {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone")

%cd {REPO_DIR}


Cloning into '/content/weathernext'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 123 (delta 27), reused 112 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 6.51 MiB | 23.88 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/weathernext


## 3. Install the model code and GPU-enabled JAX

In [6]:
!pip install -q "git+https://github.com/google-deepmind/weathernext.git@08cf73625c9d12bd9aaa038868bcb2fe488f2a22"
!pip install -q -U "jax[cuda12]"


  Preparing metadata (setup.py) ... done


## 4. Install the remaining requirements

In [5]:
!pip install -q -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 130.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 109.8 MB/s eta 0:00:00


## 5. Sanity check: does JAX see the GPU?

This should print something like `[CudaDevice(id=0)]`. If it instead shows only a CPU device, stop here -- that means JAX installed without CUDA support and needs fixing before continuing.

In [7]:
import jax
print(jax.devices())


[CudaDevice(id=0)]


## 6. Download the GenCast Mini data (~900 MB, public bucket, no auth needed)

In [8]:
!mkdir -p gencast_data/dataset gencast_data/params gencast_data/stats

!wget -q -O gencast_data/dataset/source-era5_date-2019-03-29_res-1.0_levels-13_steps-30.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/dataset/source-era5_date-2019-03-29_res-1.0_levels-13_steps-30.nc"

!wget -q -O "gencast_data/params/GenCast_1p0deg_Mini_lt2019.npz" \
  "https://storage.googleapis.com/dm_graphcast/gencast/params/GenCast%201p0deg%20Mini%20%3C2019.npz"

!wget -q -O gencast_data/stats/diffs_stddev_by_level.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/stats/diffs_stddev_by_level.nc"
!wget -q -O gencast_data/stats/mean_by_level.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/stats/mean_by_level.nc"
!wget -q -O gencast_data/stats/stddev_by_level.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/stats/stddev_by_level.nc"
!wget -q -O gencast_data/stats/min_by_level.nc \
  "https://storage.googleapis.com/dm_graphcast/gencast/stats/min_by_level.nc"

!ls -la gencast_data/dataset gencast_data/params gencast_data/stats


gencast_data/dataset:
total 684800
drwxr-xr-x 2 root root      4096 Oct  6 06:00 .
drwxr-xr-x 5 root root      4096 Oct  6 06:00 ..
-rw-r--r-- 1 root root 701219328 Dec  4  2024 source-era5_date-2019-03-29_res-1.0_levels-13_steps-30.nc

gencast_data/params:
total 224724
drwxr-xr-x 2 root root      4096 Oct  6 06:00 .
drwxr-xr-x 5 root root      4096 Oct  6 06:00 ..
-rw-r--r-- 1 root root 230105815 Dec  4  2024 GenCast_1p0deg_Mini_lt2019.npz

gencast_data/stats:
total 24
drwxr-xr-x 2 root root 4096 Oct  6 06:00 .
drwxr-xr-x 5 root root 4096 Oct  6 06:00 ..
-rw-r--r-- 1 root root 2332 Dec  4  2024 diffs_stddev_by_level.nc
-rw-r--r-- 1 root root 2516 Dec  4  2024 mean_by_level.nc
-rw-r--r-- 1 root root 2180 Dec  4  2024 min_by_level.nc
-rw-r--r-- 1 root root 2516 Dec  4  2024 stddev_by_level.nc


Check the listing above shows real file sizes (hundreds of MB for the dataset/params files, not 0 bytes) before continuing.

## 7. Run the model

Same script, same `triblockdiag_mha` GPU attention fix already verified on
the H200 GPU server -- just a single T4 here, so the ensemble stays small.
If this fails with a CUDA out-of-memory error, lower
`--num-ensemble-members` to 2 rather than retrying blindly with the same
settings.

In [9]:
!python gencast_mini_run.py --data-dir ./gencast_data --output-dir ./gencast_output --num-ensemble-members 4


06:07:19 [INFO] NumExpr defaulting to 2 threads.
06:07:31 [INFO] JAX version: 0.11.2
06:07:31 [INFO] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
06:07:32 [INFO] JAX devices: [CudaDevice(id=0)]
06:07:32 [INFO] Loading checkpoint from gencast_data/params/GenCast_1p0deg_Mini_lt2019.npz
06:07:32 [INFO] Model description: 
        GenCast model at lower, 1deg, resolution, with 13 pressure levels and a
        4 times refined icosahedral mesh. This model is trained on ERA5 data
        from 1979 to 2018, and can be causally evaluated on 2019 and later years.
        This model has the smallest memory footprint of those provided and has been provided
        to enable low cost demonstrations. It is not representative of GenCast's performance.
        
06:07:32 [INFO] Loading example dataset from gencast_data/dataset/source-era5_date-2019-03-29_res-1.0_levels-13_steps-30.nc
/content/weathernext/gen

## 8. Check the outputs

`predictions.nc` plus the sanity-check / error-map / RMSE-by-lead-time
diagnostic plots should now be in `gencast_output/` on the Colab VM.

In [1]:
!ls -la gencast_output


ls: cannot access 'gencast_output': No such file or directory


## 9. (Optional) Download results back to your laptop

The files above live on the remote Colab VM, not your laptop. If your VS
Code Colab connection doesn't already sync `/content/weathernext` back to a
local folder, zip the output and download it manually (e.g. via the Colab
file browser, or `scp`/`rsync` if your connection method supports it).

In [ ]:
!zip -r gencast_output.zip gencast_output
print("Zipped -- download gencast_output.zip from the Colab VM's file browser if it isn't already synced to your laptop.")
